# 新品香水市场潜力预测模型

这个 Notebook 的目标是构建一个新品香水市场潜力辅助评估模型。

它不是预测香水是否一定好闻，也不是预测真实销量，而是回答：

**如果一款新品香水在上市前已知品牌、国家、Gender、主香调、前调、中调和后调，它是否更接近历史高热度香水的特征？**

为了避免数据泄漏，模型不会把 `Rating Count` 当作输入特征，因为新品上市前没有评分人数。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.metrics.pairwise import cosine_similarity

data_path = "fra_cleaned.csv"
df = pd.read_csv(data_path, sep=";", encoding="cp1252", decimal=",")
df.head()


## 1. 数据清洗与目标变量构造

In [ ]:
df.columns = [c.strip() for c in df.columns]

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"": np.nan, "nan": np.nan, "unknown": np.nan, "Unknown": np.nan})

for col in ["Rating Value", "Rating Count", "Year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Popularity Score"] = df["Rating Value"] * np.log1p(df["Rating Count"])

model_df = df.dropna(subset=["Popularity Score", "Rating Value"]).copy()

threshold = model_df["Popularity Score"].quantile(0.90)
model_df["High_Potential"] = (model_df["Popularity Score"] >= threshold).astype(int)

print("High potential threshold:", round(threshold, 3))
print("High potential rate:", round(model_df["High_Potential"].mean(), 3))


## 2. 构造上市前可知特征

为了模拟新品预测场景，这里只使用上市前可以知道的信息：

Brand、Country、Gender、Year、mainaccord1 到 mainaccord5、Top、Middle、Base。

不使用 Rating Count 作为输入，否则会造成数据泄漏。

In [ ]:
accord_cols = [c for c in model_df.columns if c.startswith("mainaccord")]
note_cols = [c for c in ["Top", "Middle", "Base"] if c in model_df.columns]
text_cols = accord_cols + note_cols

def clean_text_value(x):
    if pd.isna(x):
        return ""
    return str(x).lower().replace(",", " ").replace(";", " ").replace("|", " ")

model_df["scent_text"] = ""
for col in text_cols:
    model_df["scent_text"] += " " + model_df[col].apply(clean_text_value)

model_df["scent_text"] = model_df["scent_text"].str.replace(r"\s+", " ", regex=True).str.strip()
model_df = model_df[model_df["scent_text"].str.len() > 0].copy()

top_brands = model_df["Brand"].value_counts().head(300).index
model_df["Brand_model"] = np.where(model_df["Brand"].isin(top_brands), model_df["Brand"], "Other_Brand")

feature_cols = ["Brand_model", "Country", "Gender", "Year", "scent_text"]

X = model_df[feature_cols].copy()
y_class = model_df["High_Potential"].copy()
y_reg = model_df["Popularity Score"].copy()

X.head()


## 3. 划分训练集和测试集

In [ ]:
X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

print(X_train.shape, X_test.shape)


## 4. 训练分类模型

In [ ]:
preprocess_linear = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Brand_model", "Country", "Gender"]),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), ["Year"]),
        ("text", TfidfVectorizer(stop_words="english", max_features=1200, ngram_range=(1, 2), min_df=5), "scent_text"),
    ],
    remainder="drop"
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Brand_model", "Country", "Gender"]),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), ["Year"]),
        ("text", TfidfVectorizer(stop_words="english", max_features=1200, ngram_range=(1, 2), min_df=5), "scent_text"),
    ],
    remainder="drop"
)

logit_clf = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", LogisticRegression(max_iter=600, class_weight="balanced", random_state=42))
])

rf_clf = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(
        n_estimators=80,
        max_depth=18,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

logit_clf.fit(X_train, y_train_class)
rf_clf.fit(X_train, y_train_class)

logit_pred = logit_clf.predict(X_test)
logit_proba = logit_clf.predict_proba(X_test)[:, 1]
rf_pred = rf_clf.predict(X_test)
rf_proba = rf_clf.predict_proba(X_test)[:, 1]


## 5. 评估分类模型

In [ ]:
def clf_metrics(name, y_true, y_pred, y_proba):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba)
    }

metrics_class = pd.DataFrame([
    clf_metrics("Logistic Regression Baseline", y_test_class, logit_pred, logit_proba),
    clf_metrics("Random Forest Classifier", y_test_class, rf_pred, rf_proba)
]).round(4)

metrics_class


In [ ]:
print(classification_report(y_test_class, rf_pred, zero_division=0))
confusion_matrix(y_test_class, rf_pred)


## 6. 训练连续热度分数预测模型

In [ ]:
ridge_reg = Pipeline([
    ("preprocess", preprocess_linear),
    ("model", Ridge(alpha=1.0))
])

ridge_reg.fit(X_train, y_train_reg)
reg_pred = ridge_reg.predict(X_test)

metrics_reg = pd.DataFrame([{
    "Model": "Ridge Regression",
    "MAE": mean_absolute_error(y_test_reg, reg_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_reg, reg_pred)),
    "R2": r2_score(y_test_reg, reg_pred)
}]).round(4)

metrics_reg


## 7. 新品香水概念预测函数

In [ ]:
similarity_vectorizer = TfidfVectorizer(stop_words="english", max_features=1200, ngram_range=(1, 2), min_df=5)
scent_matrix = similarity_vectorizer.fit_transform(model_df["scent_text"])

def build_input_row(
    brand="Dior",
    country="France",
    gender="women",
    year=2026,
    mainaccords="vanilla amber sweet warm spicy white floral",
    top="orange blossom pear bergamot",
    middle="coffee jasmine almond",
    base_notes="vanilla patchouli cedar"
):
    scent_text = " ".join([mainaccords, top, middle, base_notes]).lower()
    brand_model = brand if brand in set(top_brands) else "Other_Brand"
    return pd.DataFrame([{
        "Brand_model": brand_model,
        "Country": country,
        "Gender": gender,
        "Year": year,
        "scent_text": scent_text
    }])[feature_cols]

def evaluate_new_perfume(row, top_n=8):
    high_prob = rf_clf.predict_proba(row)[0, 1]
    predicted_score = ridge_reg.predict(row)[0]

    input_vec = similarity_vectorizer.transform([row["scent_text"].iloc[0]])
    sims = cosine_similarity(input_vec, scent_matrix).flatten()

    temp = model_df.copy()
    temp["Similarity"] = sims
    similar = temp.sort_values(["Similarity", "Popularity Score"], ascending=False).head(top_n)

    similar_out = similar[
        ["Perfume", "Brand", "Country", "Gender", "Rating Value", "Rating Count",
         "Popularity Score", "High_Potential", "Similarity", "Year", "scent_text"]
    ].round(3)

    similar_count = int((sims >= 0.20).sum())
    avg_top_similarity = float(np.mean(np.sort(sims)[-top_n:]))

    if similar_count >= 50 and avg_top_similarity >= 0.35:
        confidence = "High"
    elif similar_count >= 15 and avg_top_similarity >= 0.25:
        confidence = "Medium"
    else:
        confidence = "Low"

    return high_prob, predicted_score, confidence, similar_count, avg_top_similarity, similar_out

sample_input = build_input_row()
prob, score, confidence, similar_count, avg_sim, similar_cases = evaluate_new_perfume(sample_input)

print("Predicted high potential probability:", round(prob, 3))
print("Predicted popularity score:", round(score, 3))
print("Prediction confidence:", confidence)
print("Historical similar sample count:", similar_count)
print("Average top similarity:", round(avg_sim, 3))

similar_cases


## 8. 如何解释这个模型

这个模型不能说明“这款香水一定好闻”或“一定会卖爆”。它只能说明：

**这个新品概念在品牌、国家、Gender 和气味结构上，是否更接近 Fragrantica 历史高热度香水。**

为了避免热门香调随意堆叠造成误判，模型会同时返回历史相似样本和预测可信度。历史相似样本越多，Top 相似度越高，预测结果越有参考价值。